build response matrix Y for each week

then, for each week Y matrix, put it into a dataframe with cols: subj_id, week, v{i} (i in [0, p-1] where p = x * y * z)

In [ ]:
import numpy as np
import pandas as pd
import nibabel as nib
import nitools as nt

import smarts_cerebellum.globals as gl
#from smarts_cerebellum.util import (function for subj search within weeks; returns dict)

In [ ]:
def world_indices(img):
    """
    Function to get indices (x,y,z) for world coordinates for an image
    """
    i, j, k = np.indices(img.shape)
    x,y,z = nt.affine_transform(i, j, k, img.affine)
    return x,y,z

In [ ]:
def response_matrix_week(subj_path_dict,
                         x,y,z, # indices from world image
                         week, # which week this matrix is for
                         ):
    # initialize empty array
    week_subj_rows = []
    week_subj_ids = [] # extra check: store subj_id AFTER they are added to matrix
    
    for s_id, s_path in subj_path_dict.items():
        # let's do a try-except loop
        try:
            subj_img = nib.load(s_path)
            row = nt.sample_image(subj_img, 
                                  xm = x, ym = y, zm = z,
                                  interpolation = 1
                                  ).flatten() # store as row vector
        except Exception: # file path not exist; subj not added to s_id for that week
            print(f'path {s_path} not exist; skip')
            continue

        week_subj_rows.append(row)
        week_subj_ids.append(s_id)
    
    Y_w = np.array(week_subj_rows)

    return Y_w, np.array(week_subj_ids), week

In [ ]:
def make_week_dataframe(Y_w,
                   sid, 
                   week, # which week this df is for
                   template_img):
    """
    makes dataframe out of each week's response matrix
    """
    template_arr = template_img.get_fdata()
    P = np.prod(template_arr.shape)
    df = pd.DataFrame(data = Y_w, 
                      columns = [f'v{i}' for i in range(P)]
                      )
    df.insert(0, 'Week', week) # add week as beginning col
    df.insert(0, 'Subj', sid)

    return df

function to make a response dataframe

function to perform the lme (combines everything)

In [ ]:
def response_dataframe(subj_path_dict,
                       x,y,z,
                       template_img,
                       ):
    """
    Make full response dataframe
    """
    dfs = []
    time_pts = [0, 4, 12, 14, 52] # all time points
    for t in time_pts:
        Y_w, sid = response_matrix_week(subj_path_dict,
                                        x,y,z,
                                        week = t)
        
        week_df = make_week_dataframe(Y_w=Y_w, sid = sid,
                               week = t,
                               template_img = template_img
                               )
        dfs.append(week_df)

    # combine dfs by row
    Y_df = pd.concat(dfs, axis = 0, ignore_index = True)